In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
#
# Linux 환경에서 동작하는 ZIP 내부 'grayscale' 폴더 내 DICOM(.dcm) 파일만
# 추출하여 /public/sylee/Data/<병원명>/<zip파일명>/...grayscale/... 구조로 보존합니다.

import os
import zipfile
import glob
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

# ──────────────────────────────────────────────────────────────
# 설정: 원본 ZIP 파일이 들어있는 병원별 경로(리눅스 절대 경로)와 출력 베이스 폴더
# ──────────────────────────────────────────────────────────────
source_dirs = {
    "EUMC": "/public/data/MDAISS/EUMC",
    "KNUCH": "/public/data/MDAISS/KNUCH",
    "KUAH": "/public/data/MDAISS/KUAH",
    "SMC":  "/public/data/MDAISS/SMC"
}
base_output = "/public/sylee/Data"
missing = defaultdict(list)

# I/O 바운드 작업이므로 CPU 코어 × 4 만큼 쓰레드 풀 사용
max_workers = os.cpu_count() * 4

def extract_dcm_from_grayscale(zip_path, hospital):
    zip_name = os.path.basename(zip_path)
    zip_base, _ = os.path.splitext(zip_name)

    # 출력 폴더: /public/sylee/Data/<병원명>/<zip파일명>
    out_dir = os.path.join(base_output, hospital, zip_base)
    os.makedirs(out_dir, exist_ok=True)

    if not zipfile.is_zipfile(zip_path):
        print(f"[WARNING] Not a ZIP file: {zip_name}")
        return zip_name, False

    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        targets = []
        for m in members:
            # ZIP 내부 경로 구분자 '\\' → '/' 로 정규화
            posix_path = m.replace('\\', '/')
            # 'grayscale' 폴더 경로에 있고 '.dcm' 확장자일 경우만 선택
            parts = posix_path.split('/')
            if any('grayscale' in p.lower() for p in parts) and posix_path.lower().endswith('.dcm'):
                targets.append((m, posix_path))

        if not targets:
            print(f"[MISSING] No DICOM files under 'grayscale' in {zip_name} ({hospital})")
            return zip_name, False

        for raw_member, posix_path in targets:
            # 최종 경로: out_dir + posix_path (POSIX '/' 유지)
            dest_path = os.path.join(out_dir, *posix_path.split('/'))
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            # 파일 내용 스트림 복사
            with zf.open(raw_member) as src, open(dest_path, 'wb') as dst:
                dst.write(src.read())

        print(f"[DONE] Extracted {len(targets)} DICOM(s) from {zip_name} to {hospital}/{zip_base}")
        return zip_name, True

# ──────────────────────────────────────────────────────────────
# 병렬 처리 실행
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = []
        for hosp, src in source_dirs.items():
            pattern = os.path.join(src, "*.zip")
            for zp in glob.glob(pattern):
                futures.append(pool.submit(extract_dcm_from_grayscale, zp, hosp))

        for fut in futures:
            name, ok = fut.result()
            if not ok:
                hospital = name.split('_', 1)[0]
                missing[hospital].append(name)

    # ──────────────────────────────────────────────────────────────
    # 누락 리포트 작성
    # ──────────────────────────────────────────────────────────────
    report_path = os.path.join(base_output, "missing_dcm_report.txt")
    with open(report_path, 'w') as report:
        for hosp, files in sorted(missing.items()):
            report.write(f"{hosp}: {', '.join(files)}\n")

    print(f"All ZIP processing done. Missing-report → {report_path}")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import json

import numpy as np
import ants
import nibabel as nib
import pydicom
from pydicom.multival import MultiValue

BASE_DIRS = [
    "/public/sylee/Data/EUMC",
    "/public/sylee/Data/KNUCH",
    "/public/sylee/Data/KUAH",
    "/public/sylee/Data/SMC",
]
OUTPUT_ROOT = "/public/sylee/MDAISS_2/output/nifit"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

def get_folder_components(folder_path):
    parts = folder_path.rstrip(os.sep).split(os.sep)
    return parts[-3], parts[-2], parts[-1]

def dicom_dataset_to_dict(ds):
    """
    pydicom Dataset의 모든 DataElement를 순회하며
    JSON 직렬화 가능한 dict로 변환.
    """
    meta = {}
    for elem in ds:
        key = elem.keyword if elem.keyword else str(elem.tag)
        val = elem.value

        # 직렬화 가능하게 변환
        if isinstance(val, MultiValue):
            meta[key] = list(val)
        elif isinstance(val, (list, tuple)):
            meta[key] = [v.tolist() if hasattr(v, "tolist") else v for v in val]
        elif isinstance(val, (str, int, float, bool)):
            meta[key] = val
        else:
            meta[key] = str(val)
    return meta

def process_grayscale_folder(folder):
    # 1) ANTs로 DICOM 시리즈 읽기 (볼륨만)
    img = ants.image_read(folder)
    vol = img.numpy().astype(np.float32)  # (H, W, D)

    # 2) 0–1 정규화
    vmin, vmax = vol.min(), vol.max()
    vol = (vol - vmin) / (vmax - vmin + 1e-8)

    # 3) XY plane 정사각형 패딩
    h, w, d = vol.shape
    M = max(h, w)
    pad_h, pad_w = M - h, M - w
    pads = (
        (pad_h // 2, pad_h - pad_h // 2),
        (pad_w // 2, pad_w - pad_w // 2),
        (0, 0)
    )
    vol_padded = np.pad(vol, pads, mode="constant", constant_values=0.0)

    # 4) 메타데이터: 첫 DICOM 파일을 pydicom으로 읽어서 전부 추출
    first_dcm = sorted(glob.glob(os.path.join(folder, "*.dcm")))[0]
    ds = pydicom.dcmread(first_dcm, force=True)
    meta = dicom_dataset_to_dict(ds)

    # 5) 출력 파일명 구성
    hosp, study, series = get_folder_components(folder)
    base = f"{hosp}_{study}_{series}"
    json_path = os.path.join(OUTPUT_ROOT, base + ".json")
    nii_path  = os.path.join(OUTPUT_ROOT, base + ".nii.gz")

    # 6) JSON 쓰기
    with open(json_path, "w") as jf:
        json.dump(meta, jf, indent=4, ensure_ascii=False)

    # 7) NIfTI 저장 (identity affine)
    nib_img = nib.Nifti1Image(vol_padded, affine=np.eye(4))
    nib.save(nib_img, nii_path)

    print(f"[OK] {base}")

if __name__ == "__main__":
    for base in BASE_DIRS:
        pattern = os.path.join(base, "*", "*_grayscale")
        for folder in glob.glob(pattern):
            process_grayscale_folder(folder)
